In [9]:
import re

def generate_heuristic_latex_tables(file_list):
    """
    Analyse les fichiers de log et génère le code LaTeX pour deux tableaux.
    Ordre attendu : [basique, multicamp, optxety, rf]
    Un \hline est ajouté uniquement lors du changement de groupe d'instances (ex: data0 -> data1).
    """
    all_data = {}

    # Regex pour capturer : Nom, Scénario (s), Valeur (bd), Gap (%), et Temps
    pattern = re.compile(
        r"(\w+\.txt)\s+s\s*=\s*(\d+).*?bd\s*=\s*([\d\.-]+).*?gap\s*=\s*([\d\.-]+)%.*?(?:CPU|cpu|secondes)\s*=\s*([\d\.]+)",
        re.IGNORECASE
    )

    # Traitement des cas "irréalisable"
    irrealisable_pattern = re.compile(r"(\w+\.txt)\s+s\s*=\s*(\d+).*?irréalisable", re.IGNORECASE)

    for file_path in file_list:
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                content = f.read().replace('\n', ' ')
                
                # Données standards
                matches = pattern.findall(content)
                for (data, s, bd, gap, cpu) in matches:
                    key = (data.replace('.txt', ''), s)
                    if key not in all_data:
                        all_data[key] = {}
                    all_data[key][file_path] = {"VAL": bd, "Gap": f"{gap}\\%", "CPU": cpu}
                
                # Données irréalisables
                irr_matches = irrealisable_pattern.findall(content)
                for (data, s) in irr_matches:
                    key = (data.replace('.txt', ''), s)
                    if key not in all_data:
                        all_data[key] = {}
                    all_data[key][file_path] = {"VAL": "Irréal.", "Gap": "100\%", "CPU": "--"}
        except FileNotFoundError:
            print(f"Erreur : Le fichier {file_path} est introuvable.")

    def build_table(h1_file, h2_file, h1_name, h2_name):
        latex = [
            "\\begin{table}[h!tbp]",
            "    \\centering\\small",
            "    \\renewcommand{\\arraystretch}{1.1}",
            "    \\begin{tabular}{|c|c||c|c|c||c|c|c|}",
            "        \\hline",
            f"        \\multicolumn{{2}}{{|c||}}{{\\textbf{{Instances}}}} & \\multicolumn{{3}}{{|c||}}{{\\textbf{{Heuristique {h1_name}}}}} & \\multicolumn{{3}}{{|c|}}{{\\textbf{{Heuristique {h2_name}}}}} \\\\",
            "        \\hline",
            "        \\textbf{Data} & \\textbf{S} & \\textbf{VAL} & \\textbf{Gap} & \\textbf{CPU\\_2 (s)} & \\textbf{VAL} & \\textbf{Gap} & \\textbf{CPU\\_2 (s)} \\\\",
            "        \\hline"
        ]

        sorted_keys = sorted(all_data.keys(), key=lambda x: (x[0], int(x[1])))
        
        last_file_prefix = None
        
        for i, (data, s) in enumerate(sorted_keys):
            # Extraction du préfixe (ex: 'data0' de 'data0.txt' ou 'toy')
            # On retire les chiffres à la fin pour grouper (optionnel) ou on garde le nom exact
            current_prefix = re.sub(r'\d+$', '', data) if "data" in data else data
            
            # Si le nom de fichier change par rapport au précédent, on tire un trait
            if last_file_prefix is not None and data != last_file_prefix:
                latex.append("        \\hline")
            
            last_file_prefix = data
            
            metrics = all_data[(data, s)]
            m1 = metrics.get(h1_file, {"VAL": "--", "Gap": "--", "CPU": "--"})
            m2 = metrics.get(h2_file, {"VAL": "--", "Gap": "--", "CPU": "--"})
            
            line = f"        {data} & {s} & {m1['VAL']} & {m1['Gap']} & {m1['CPU']} & {m2['VAL']} & {m2['Gap']} & {m2['CPU']} \\\\"
            latex.append(line)
        
        latex.append("        \\hline") # Ligne finale du tableau
        latex.extend([
            "    \\end{tabular}",
            f"    \\caption{{Comparaison des performances : {h1_name} vs {h2_name}}}",
            "    \\label{tab:" + h1_name.lower() + "_" + h2_name.lower() + "}",
            "\\end{table}\n"
        ])
        return "\n".join(latex)

    # Table 1: Basique vs MultiCamp (Fichiers 0 et 1)
    table1 = build_table(file_list[0], file_list[1], "Basique", "MultiCamp")
    
    # Table 2: OptXetY vs RF (Fichiers 2 et 3)
    table2 = build_table(file_list[2], file_list[3], "OptXetY", "RF")
    
    return table1 + "\n" + table2

# --- CONFIGURATION ET EXECUTION ---
# Assurez-vous que les fichiers sont dans le même répertoire ou modifiez les chemins
files = [
    "../output/gap_heur_basique.txt",    # Index 0
    "../output/gap_heur_multiCamp.txt",  # Index 1
    "../output/gap_heur_optXetY.txt",    # Index 2
    "../output/gap_heur_rf.txt"          # Index 3
]

latex_output = generate_heuristic_latex_tables(files)
print(latex_output)

# Optionnel : Sauvegarder dans un fichier .tex
with open("../output/tableaux_heur_resultats.tex", "w", encoding="utf-8") as f:
    f.write(latex_output)

\begin{table}[h!tbp]
    \centering\small
    \renewcommand{\arraystretch}{1.1}
    \begin{tabular}{|c|c||c|c|c||c|c|c|}
        \hline
        \multicolumn{2}{|c||}{\textbf{Instances}} & \multicolumn{3}{|c||}{\textbf{Heuristique Basique}} & \multicolumn{3}{|c|}{\textbf{Heuristique MultiCamp}} \\
        \hline
        \textbf{Data} & \textbf{S} & \textbf{VAL} & \textbf{Gap} & \textbf{CPU\_2 (s)} & \textbf{VAL} & \textbf{Gap} & \textbf{CPU\_2 (s)} \\
        \hline
        data0 & 0 & 8648214425640.0 & -0.44\% & 0.673 & 8624316633649.2 & -0.17\% & 0.604 \\
        data0 & 1 & 8884930649640.0 & -0.43\% & 0.642 & 8861032857649.2 & -0.16\% & 0.627 \\
        \hline
        data1 & 0 & 191362312988.98526 & -12.24\% & 9.149 & 183098249395.8927 & -7.39\% & 8.417 \\
        data1 & 1 & 191362312988.98526 & -9.44\% & 9.732 & 173534953881.04657 & -4.86\% & 8.521 \\
        data1 & 2 & 178432906076.40906 & -9.57\% & 9.816 & 169279358587.73328 & -3.95\% & 8.559 \\
        data1 & 3 & 164675453288

<>:4: SyntaxWarning: invalid escape sequence '\h'
<>:39: SyntaxWarning: invalid escape sequence '\%'
<>:4: SyntaxWarning: invalid escape sequence '\h'
<>:39: SyntaxWarning: invalid escape sequence '\%'
/tmp/ipykernel_346532/3952637651.py:4: SyntaxWarning: invalid escape sequence '\h'
  """
/tmp/ipykernel_346532/3952637651.py:39: SyntaxWarning: invalid escape sequence '\%'
  all_data[key][file_path] = {"VAL": "Irréal.", "Gap": "100\%", "CPU": "--"}


In [1]:
import pandas as pd
import io



In [3]:
latex_data = r"""\begin{longtable}{|c|c|c|c|c|c|}
\caption{Énergie produite par pas de temps} \\
\hline
\textbf{Pas de temps} & \textbf{Gaz} & \textbf{Charbon} & \textbf{Nucl 1} & \textbf{Nucl 2} & Demande \\ 
\hline
\endfirsthead

\hline
\textbf{Pas de temps} & \textbf{Gaz} & \textbf{Charbon} & \textbf{Nucl 1} & \textbf{Nucl 2} & Demande \\ 
\hline
\endhead

\hline
\endfoot

\hline
\endlastfoot
0 & 70 & 80 & 0 &0 & 150 \\ \hline
1 & 90 & 80 & 0 & 0 & 170 \\ \hline
2 & 100 & 80 & 0 & 50& 230\\ \hline
3 & 100 & 80 & 0 &30& 210\\ \hline
4 & 100 & 80 & 0 & 10& 190\\ \hline
5 & 60 & 80 & 0 & 0 & 140\\ \hline
6 & 100 & 80 & 0 &40 & 220\\ \hline
7 & 100 & 80 & 0 & 10 & 190 \\ \hline
8 & 100 & 80 & 0 & 20  &200 \\ \hline
9 & 0 & 80 & 10& 140 & 230 \\ \hline
10 & 100 & 80 & 0 & 60 & 240 \\ \hline
11 & 100 &80 & 0 & 70 & 250 \\ \hline
12 & 100 & 80  & 47.0 & 2.916 & 230 \\ \hline
13 & 100 & 80 & 10 & 0 & 190 \\ \hline
14 & 70 & 80 & \textcolor{red}{0} & 0 & 150 \\ \hline
15 & 90 & 80& \textcolor{red}{0} & 0 & 170 \\ \hline
16 & 100 & 80 & \textcolor{red}{0} & 50 & 230 \\ \hline
17 & 0 & 80 &\textcolor{red}{0}  & 130 & 210 \\ \hline
18 & 100 & 80 & \textcolor{red}{0} & 10 & 190 \\ \hline
19 & 60 & 80 & \textcolor{red}{0} & 0 & 140 \\ \hline
20 & 100 & 80 & \textcolor{red}{0} & 40 & 220 \\ \hline
21 & 100 & 80& \textcolor{red}{0} & 10 & 190 \\ \hline
22 & 100 & 80 & \textcolor{red}{0} & 20 & 200 \\ \hline
23 & 100 & 80 & \textcolor{red}{0} & 50 & 230 \\ \hline
24 & 100 & 80 & \textcolor{red}{0} & 60 & 240 \\ \hline
25 & 100 & 80 & \textcolor{red}{0} & 70& 250 \\ \hline
26 & 100 & 80 & \textcolor{red}{0} & 50 & 230 \\ \hline
27 & 100 & 80 & \textcolor{red}{0} & 10 & 190 \\ \hline
28 & 70 & 80 & 0 & 0 & 150 \\ \hline
29 & 90 & 80 & 0  & 0 & 170  \\ \hline
30 & 100 & 80 & 50 & 0 & 230   \\ \hline
31 & 100 & 80 & 30 & 0 & 210  \\ \hline
32 & 100 & 80 & 10 & 0 & 190 \\ \hline
33 & 60 & 80  & 0 & 0 & 140 \\ \hline
34 & 100 & 80  & 40 & 0 & 220 \\ \hline
35 & 100 & 80 & 10 & 0 & 190 \\ \hline
36 & 100 & 80 & 20 & 0 & 200 \\ \hline
37 & 100 & 80 & 50 & 0 & 230 \\ \hline
38 & 100 & 80 & 60 & 0 & 240 \\ \hline
39 & 100 & 80 & 70 &0 & 250 \\ \hline
40 & 100 & 80 & 50 & 0 & 230 \\ \hline
41 & 100 & 80 & 10 & 0 & 190 \\ \hline
42 & 70  & 80 & 0 & \textcolor{red}{0} & 150 \\ \hline
43 & 90  & 80 & 0 & \textcolor{red}{0} & 170 \\ \hline
44 & 100 & 80 & 50 & \textcolor{red}{0} & 230 \\ \hline
45 & 100 & 80 & 30 & \textcolor{red}{0} & 210 \\ \hline
46 & 100 & 80 & 10 & \textcolor{red}{0} & 190 \\ \hline
47 & 60 & 80 & 0 & \textcolor{red}{0} & 140 \\ \hline
48 & 100 & 80 & 40 & \textcolor{red}{0} & 220 \\ \hline
49 & 100 & 80 & 10 & \textcolor{red}{0} & 190 \\ \hline
50 & 100 & 80 & 20 & \textcolor{red}{0} & 200 \\ \hline
51 & 100 & 80 & 50 & \textcolor{red}{0} & 230 \\ \hline
52 & 100 & 80 & 60 & \textcolor{red}{0} & 240 \\ \hline
53 & 100 & 80 & 70 & \textcolor{red}{0} & 250 \\ \hline
54 & 100 & 80 & 50 & \textcolor{red}{0} & 230 \\ \hline
55 & 100 & 80 & 10 & \textcolor{red}{0} & 190 \\ \hline
56 & 70  & 80 & 0  & 0 & 150 \\ \hline
57 & 90  & 80 & 0  & 0 & 170 \\ \hline
58 & 100 & 80 & 50 & 0 & 230 \\ \hline
59 & 0  & 80 & 130 & 0 & 210 \\ \hline
60 & 100 & 80 & 10 & 0 & 190 \\ \hline
61 & 0   & 80 & 60 & 0 & 140 \\ \hline
62 & 100 & 80 & 0  & 40 & 220 \\ \hline
63 & 100 & 80 & 10 & 0 & 190 \\ \hline
64 & 100 & 80 & 20 & 0 & 200 \\ \hline
65 & 100 & 80 & 50 & 0 & 230 \\ \hline
66 & 100 & 80 & 60 & 0 & 240 \\ \hline
67 & 100 & 80 & 70 & 0 & 250 \\ \hline
68 & 100 & 80 & 50 & 0 & 230 \\ \hline
69 & 100 & 80 & 10 & 0 & 190 \\ \hline
70 & 0   & 80 & 0  & 70 & 150 \\ \hline
71 & 90  & 80 & 0  & 0 & 170 \\ \hline
72 & 100 & 80 & 0  & 50 & 230 \\ \hline
73 & 100 & 80 & 0  & 30 & 210 \\ \hline
74 & 100 & 80 & 10 & 0 & 190 \\ \hline
75 & 0   & 80 & 60 & 0 & 140 \\ \hline
76 & 100 & 80 & 40 & 0 & 220 \\ \hline
77 & 100 & 80 & 0  & 10 & 190 \\ \hline
78 & 100 & 80 & 20 & 0 & 200 \\ \hline
79 & 100 & 80 & 0  & 50 & 230 \\ \hline
80 & 100 & 80 & 60 & 0 & 240 \\ \hline
81 & 100 & 80 & 37.92 & 32.08 & 250 \\ \hline
82 & 100 & 80 & 50 & 0 & 230 \\ \hline
83 & 100 & 80 & 10 & 0 & 190 \\ \hline
84 & 0   & 80 & 70 & 0 & 150 \\ \hline
85 & 90  & 80 & 0  & 0 & 170 \\ \hline
86 & 100 & 80 & 50 & 0 & 230 \\ \hline
87 & 100 & 80 & 30 & 0 & 210 \\ \hline
88 & 100 & 80 & 10 & 0 & 190 \\ \hline
89 & 0   & 80 & 60 & 0 & 140 \\ \hline
90 & 100 & 80 & 40 & 0 & 220 \\ \hline
91 & 100 & 80 & 10 & 0 & 190 \\ \hline
92 & 100 & 80 & 20 & 0 & 200 \\ \hline
93 & 100 & 80 & 50 & 0 & 230 \\ \hline
94 & 100 & 80 & \textcolor{red}{0}  & 60 & 240 \\ \hline
95 & 100 & 80 & \textcolor{red}{0}  & 70 & 250 \\ \hline
96 & 100 & 80 & \textcolor{red}{0} & 50 & 230 \\ \hline
97 & 100 & 80 & \textcolor{red}{0}  & 10 & 190 \\ \hline
98 & 70  & 80 & \textcolor{red}{0}  & 0 & 150 \\ \hline
99 & 90  & 80 & \textcolor{red}{0}  & 0 & 170 \\ \hline
100 & 100 & 80 & \textcolor{red}{0}  & 50 & 230 \\ \hline
101 & 100 & 80 & \textcolor{red}{0}  & 30 & 210 \\ \hline
102 & 100 & 80 & \textcolor{red}{0} & 10 & 190 \\ \hline
103 & 60  & 80 & \textcolor{red}{0}  & 0 & 140 \\ \hline
104 & 100 & 80 & \textcolor{red}{0}  & 40 & 220 \\ \hline
105 & 100 & 80 & \textcolor{red}{0}  & 10 & 190 \\ \hline
106 & 100 & 80 & \textcolor{red}{0}  & 20 & 200 \\ \hline
107 & 100 & 80 & \textcolor{red}{0} & 50 & 230 \\ \hline
108 & 100 & 80 & 60 & 0 & 240 \\ \hline
109 & 100 & 80 & 70 & 0 & 250 \\ \hline
110 & 100 & 80 & 50 & 0 & 230 \\ \hline
111 & 100 & 80 & 10 & 0 & 190 \\ \hline
112 & 70  & 80 & 0  & 0 & 150 \\ \hline
113 & 90  & 80 & 0  & 0 & 170 \\ \hline
114 & 100 & 80 & 50 & 0 & 230 \\ \hline
115 & 100 & 80 & 30 & 0 & 210 \\ \hline
116 & 100 & 80 & 10 & 0 & 190 \\ \hline
117 & 60  & 80 & 0  & 0 & 140 \\ \hline
118 & 100 & 80 & 40 & \textcolor{red}{0} & 220 \\ \hline
119 & 100 & 80 & 10 & \textcolor{red}{0} & 190 \\ \hline
120 & 100 & 80 & 20 & \textcolor{red}{0} & 200 \\ \hline
121 & 100 & 80 & 50 & \textcolor{red}{0} & 230 \\ \hline
122 & 100 & 80 & 60 & \textcolor{red}{0} & 240 \\ \hline
123 & 100 & 80 & 70 & \textcolor{red}{0} & 250 \\ \hline
124 & 100 & 80 & 50 & \textcolor{red}{0} & 230 \\ \hline
125 & 0   & 80 & 110 & \textcolor{red}{0} & 190 \\ \hline
126 & 70  & 80 & 0  & \textcolor{red}{0} & 150 \\ \hline
127 & 90  & 80 & 0  & \textcolor{red}{0} & 170 \\ \hline
128 & 100 & 80 & 50 & \textcolor{red}{0} & 230 \\ \hline
129 & 100 & 80 & 30 & \textcolor{red}{0} & 210 \\ \hline
130 & 100 & 80 & 10 & \textcolor{red}{0} & 190 \\ \hline
131 & 60  & 80 & 0  & \textcolor{red}{0} & 140 \\ \hline
132 & 0   & 80 & 0  & 140 & 220 \\ \hline
133 & 100 & 80 & 0  & 10 & 190 \\ \hline
134 & 35  & 80 & 0  & 85 & 200 \\ \hline
135 & 10  & 80 & 0  & 140 & 230 \\ \hline
136 & 100 & 80 & 0  & 60 & 240 \\ \hline
137 & 30  & 80 & 0  & 140 & 250 \\ \hline
138 & 100 & 80 & 0  & 50 & 230 \\ \hline

\end{longtable}"""


In [6]:
import re
import csv

def latex_to_csv(latex_string, output_filename):
    clean_text = re.sub(r'\\begin\{.*\}|\\end\{.*\}|\\hline|\\caption\{.*\}|\\end.*head|\\end.*foot', '', latex_string)

    clean_text = re.sub(r'\\[a-z]+(?:\{[a-z]+\})?\{([^}]+)\}', r'\1', clean_text)

    rows = clean_text.strip().split('\\\\')

    final_data = []
    for row in rows:
        columns = [col.strip() for col in row.split('&')]
        
        if any(columns):
            final_data.append(columns)

    with open(output_filename, 'w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerows(final_data)

    print(f"Success! Data saved to {output_filename}")

latex_to_csv(latex_data, '../output/energie_produite.csv')

Success! Data saved to ../output/energie_produite.csv


In [7]:
df = pd.read_csv('../output/energie_produite.csv', sep=',', encoding='utf-8')


In [13]:
df

,Pas de temps,Gaz,Charbon,Nucl 1,Nucl 2,Demande
0,Pas de temps,Gaz,Charbon,Nucl 1,Nucl 2,Demande
1,0,70,80,0,0,150
2,1,90,80,0,0,170
3,2,100,80,0,50,230
4,3,100,80,0,30,210
...,...,...,...,...,...,...
135,134,35,80,0,85,200
136,135,10,80,0,140,230
137,136,100,80,0,60,240
138,137,30,80,0,140,250


In [14]:
def extract_prod_data_from_df(df):
    df = df.copy()
    norm = lambda s: re.sub(r'\W+', '', str(s)).lower()
    col_norm_map = {col: norm(col) for col in df.columns}
    df.rename(columns={col: col_norm_map[col] for col in df.columns}, inplace=True)

    time_col = None
    for c in df.columns:
        if 'pas' in c or 'time' in c or 'temps' in c:
            time_col = c
            break
    if time_col is not None:
        df = df[pd.to_numeric(df[time_col], errors='coerce').notna()]

    expected = {4: 'gaz', 3: 'charbon', 2: 'nucl1', 1: 'nucl2'}
    prod_data = {}
    for key, expected_name in expected.items():
        if expected_name not in df.columns:
            avail = list(df.columns)
            raise KeyError(f"column '{expected_name}' not found after normalization. available: {avail}")
        prod_data[key] = pd.to_numeric(df[expected_name], errors='coerce').fillna(0).astype(float).tolist()

    return prod_data

df.columns = [re.sub(r'\s+', '', c) if isinstance(c, str) else c for c in df.columns]
prod_data = extract_prod_data_from_df(df)
print({k: len(v) for k, v in prod_data.items()})

{4: 139, 3: 139, 2: 139, 1: 139}


In [15]:
prod_data

{4: [70.0,
  90.0,
  100.0,
  100.0,
  100.0,
  60.0,
  100.0,
  100.0,
  100.0,
  0.0,
  100.0,
  100.0,
  100.0,
  100.0,
  70.0,
  90.0,
  100.0,
  0.0,
  100.0,
  60.0,
  100.0,
  100.0,
  100.0,
  100.0,
  100.0,
  100.0,
  100.0,
  100.0,
  70.0,
  90.0,
  100.0,
  100.0,
  100.0,
  60.0,
  100.0,
  100.0,
  100.0,
  100.0,
  100.0,
  100.0,
  100.0,
  100.0,
  70.0,
  90.0,
  100.0,
  100.0,
  100.0,
  60.0,
  100.0,
  100.0,
  100.0,
  100.0,
  100.0,
  100.0,
  100.0,
  100.0,
  70.0,
  90.0,
  100.0,
  0.0,
  100.0,
  0.0,
  100.0,
  100.0,
  100.0,
  100.0,
  100.0,
  100.0,
  100.0,
  100.0,
  0.0,
  90.0,
  100.0,
  100.0,
  100.0,
  0.0,
  100.0,
  100.0,
  100.0,
  100.0,
  100.0,
  100.0,
  100.0,
  100.0,
  0.0,
  90.0,
  100.0,
  100.0,
  100.0,
  0.0,
  100.0,
  100.0,
  100.0,
  100.0,
  100.0,
  100.0,
  100.0,
  100.0,
  70.0,
  90.0,
  100.0,
  100.0,
  100.0,
  60.0,
  100.0,
  100.0,
  100.0,
  100.0,
  100.0,
  100.0,
  100.0,
  100.0,
  70.0,
  90.0,
  100.0,

In [16]:
output_path = '../output/prod_data.txt'
with open(output_path, 'w', encoding='utf-8') as f:
    for unit_id, values in prod_data.items():
        f.write(f"{unit_id}: {values}\n")

print(f"prod_data saved to {output_path}")

prod_data saved to ../output/prod_data.txt
